In [ ]:
# Magic formulae so that changes in the .py files are loaded in here
%load_ext autoreload
%autoreload 2
###

import numpy as np
import matplotlib.pyplot as plt
import matplotlib 
import pandas as pd
import xarray as xr
import seaborn as sns
import torch
from torch import nn
from tqdm.notebook import tqdm
import sklearn
import copy


# STEP 1 PREPROCESSING DATA
The first step of the machine learning pipeline is to load the data from a file. Here, our data is stored in the `dataset_energy.csv`. We use the `pandas` library to load the data.

In [ ]:
df = pd.read_csv('../../data/dataset_energy.csv', index_col=0)
df

Our dataset contains hourly values from 2020 to 2025.

Once the data is loaded, we need to turn the datasets into an ML friendly array. To do this, we choose the variables necessary for the analysis and then we transform them into numpy arrays.

In [ ]:
X = df[['Pr','Tmin','v_mean','v_max']].values # X is the array containing the data we use to predict the result.
y = df[['Consuption Cabins and holiday properties']].values # y is the array representing the values we want to predict.

## **QUESTION 1**: Create train, validation and test datasets from `X` and `y`.

In [ ]:
X_train, X_valtest, y_train, y_valtest = sklearn.model_selection.train_test_split(X,y, train_size=0.7)
X_val, X_test, y_val, y_test = sklearn.model_selection.train_test_split(X_valtest,y_valtest, train_size=0.7)

Once we have made the split into train and test, we will transform the data using StandardScaler, which means we remove the mean and divide by the standard deviation.

## **QUESTION 2**: Transform `X` and `y` using `StandardScaler`. 
Remember, the scaler should only fit the train dataset.

In [ ]:
scaler_X = sklearn.preprocessing.StandardScaler()
scaler_y = sklearn.preprocessing.StandardScaler()

scaler_X.fit(X_train)
scaler_y.fit(y_train)

X_train = scaler_X.transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)

y_train = scaler_y.transform(y_train)
y_val = scaler_y.transform(y_val)
y_test = scaler_y.transform(y_test)


> **_CAREFUL:_**  **Pytorch does not work with numpy arrays**.
> 
> `X` and `y` datasets must be converted to tensor, which is a kind of array with additional properties.

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float)
X_val = torch.tensor(X_val, dtype=torch.float)
X_test = torch.tensor(X_test, dtype=torch.float)

y_train = torch.tensor(y_train, dtype=torch.float)
y_val = torch.tensor(y_val, dtype=torch.float)
y_test = torch.tensor(y_test, dtype=torch.float)

> In addition, we will want to use, which is a specialized computing unit which allows for faster training. We will talk about it later in the session.
>
> To use the GPU, all data and models must **be** on the GPU. We send data around by using data.to(device).
> 
> We use a device called `cuda`, which corresponds to GPU built by the NVIDIA brand.

In [ ]:
device = 'cuda'

X_train = X_train.to(device)
X_val = X_val.to(device)
X_test = X_test.to(device)

y_train = y_train.to(device)
y_val = y_val.to(device)
y_test = y_test.to(device)

# STEP 2: Define ML model and fit it to the data - Source file


There are multiple ways to define neural networks, but the standard one is to build a python class. We import this class, and two functions from the `neuralnetworks.py` file.

In [ ]:
from neuralnetworks import MultiLayerPerceptron3layers
from neuralnetworks import training_one_epoch, evaluation_one_epoch,training_one_batch

## Define hyperparameters for training
We can choose some parameters for training, including learning rate, batch size and the number of epochs. We will get back to that.

We also define an optimizer and a loss function, which is what allows to train the model.

In [ ]:
n_features = 4
n_predictions = 1
model = MultiLayerPerceptron3layers(number_input_features=n_features, number_of_predictions=n_predictions)
model = model.to(device)


# training hyperparameters
n_epochs = 20  # number of epochs to run
batch_size = 256  # size of each batch
learning_rate = 0.001  # learning rate
index_where_batches_start = torch.arange(0, len(X_train), batch_size)

loss_fn = nn.MSELoss()  # mean square error
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

model

# Train network
Training happens in different steps.

In [ ]:
# We keep the weights and biases of the best model
best_loss_value = np.inf   # initialise to infinity
best_weights = None
loss_history = []

# training loop
for epoch in tqdm(range(n_epochs)):
    model.train()
    for start in index_where_batches_start:
    # Select the batch of data
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        training_one_batch(model, X_batch, y_batch, optimizer, loss_fn)
    model.eval()
    best_loss_value, best_weights = evaluation_one_epoch(model, X_val, y_val, 
                                                         loss_fn, loss_history, 
                                                         best_loss_value, best_weights)

# restore model weights to the best model
model.load_state_dict(best_weights)
model = model.eval()


###  Plot training history: verify that the loss function goes down

In [ ]:
with torch.no_grad():
    best_pred = model(X_val).detach().cpu()


fix, ax = plt.subplots()
# ax.set_ylim(0,None)
print("MSE: %.2f" % best_loss_value)
print("RMSE: %.2f" % np.sqrt(best_loss_value))
ax.plot(loss_history)
ax.set_ylabel('Loss function (MSE)')
ax.set_xlabel('Epoch')
ax.set_xlim(0,None)
ax.grid()
plt.show()

# STEP 3: Make predictions and evaluate the model skill
Once the model is **trained** with the data, we can use it to make predictions.

## Step 3.1: Predict `y_val` from `X_val`
We predict the values of `y` from `X` based on the relation learned by the model. For neural networks with pytorch, there is no `model.predict` method. We only use `model(X)`

## **QUESTION 3**: Use the **validation data** to make predictions.

In [ ]:
y_val_predictions = model(X_val)

## Step 3.2: Plot the results and evaluate the model.
To visually evaluate the model, we plot the predicted `y` values against the true `y` values.


> This part is a bit confusing: pytorch tensors cannot be used directly for plots.
>
> Remember how we moved the data to the GPU in the first place? We have to move it back to the "normal" computer unit, the cpu.
> This is done by doing `data.to('cpu')`.
>
> Tensors also have additional properties that can only be stored on GPU and that differentiate them from normal numpy arrays. We also have to remove these additional properties before moving data back to the cpu by doing `data.detach()`.
>
> Overall, we need to write `data_cpu = data.detach().to('cpu')`
>
> Note that we also have to put the y_val back to the 'cpu'.

In [ ]:
y_val_predictions_cpu = y_val_predictions.detach().to('cpu')
y_val_cpu = y_val.detach().to('cpu')

Once the predictions are ready to be used, we need to transform them back into the original dataspace.

## **QUESTION 4**: Use the `inverse_transform` to return `y_val_predictions_cpu` and `y_val_cpu` to the original data space.

In [ ]:
y_val_predictions_cpu = scaler_y.inverse_transform(y_val_predictions_cpu)
y_val_cpu = scaler_y.inverse_transform(y_val_cpu)

### Plot the predictions against the true data for the validation

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(y_val_cpu,y_val_predictions_cpu, s=5, color='.7') 
ax.axline((y_val_cpu.mean(), y_val_cpu.mean()), slope=1, ls='--', color='tab:red')
ax.grid()

## **QUESTION 5**: Evaluate the model with the `r2_score` and the `root_mean_square_error`.

In [ ]:
rmse = sklearn.metrics.root_mean_squared_error(y_val_cpu, y_val_predictions_cpu)
r2_score = sklearn.metrics.r2_score(y_val_cpu, y_val_predictions_cpu)
print(f"RMSE: {rmse:.02f}")
print(f"R2: {r2_score:.02f}")

## **QUESTION 5**: compare the results with the linear regression
Usually, the neural netwroks will perform better for more complex relationship and non-linear tasks.

<div class="alert alert-block alert-warning">
<b>Note:</b> It is always great to compare neural network results with a much simpler linear model
</div>